# GWNet-Lean v2 — PEMS08 | Debugged + Upgraded | Target: MAE < 12.5

## Bugs fixed from ab2
1. **Cell-7 cfg crash** — `adj_thres` and `topk_graph` were used in `load_pems08` but never defined in `Config`
2. **Dead `composite_loss`** — defined but `train_epoch` used `masked_mae` instead; now wired in
3. **Weak skip aggregation** — `skip[:,:,:,-1:]` (single step) → mean of last 4 timesteps

## Architecture upgrade (needed to break 13 MAE)
The original d=64 / L=6 / seq=12 architecture has a MAE ceiling of ~13-14.
The upgrade below is the minimum change to reliably get below 13:

| Param | Before | After | Reason |
|---|---|---|---|
| `seq_len` | 12 | 24 | 2h history — single biggest accuracy gain |
| `d_model` | 64 | 96 | More channel capacity |
| `d_skip` | 256 | 384 | Richer skip aggregation |
| `n_layers` | 6 | 8 | Wider receptive field (30→60 steps) |
| `dropout` | 0.20 | 0.15 | Slightly less aggressive with bigger model |
| `weight_decay` | 5e-4 | 3e-4 | Tuned for new size |
| skip agg | last 1 | mean last 4 | Richer temporal context |
| loss | masked_mae only | composite (0.6·MAE+0.3·Huber+0.1·RMSE) | Smoother gradients |

**Estimated params: ~1.27M | Overfit ratio: ~110× (PEMS08 train ~11.5k samples)**
**Expected MAE: 11.5–12.5 | Expected RMSE: 19–21**


In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"Seed set: {seed} — reproducible ✓")

set_seed()
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    :", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")


In [ ]:
class Config:
    data_path    = "/kaggle/input/datasets/piyush1718s/pems08/PEMS08.npz"
    adj_csv_path = "/kaggle/input/datasets/piyush1718s/pems08csv/PEMS08.csv"
    num_nodes    = 170
    in_features  = 3
    seq_len      = 24    # BUG FIX + UPGRADE: 12→24 (2h history, biggest accuracy gain)
    pred_len     = 12
    feature_idx  = 0
    noise_std    = 0.012 # Gaussian noise augmentation on normalised inputs (tuned down for larger model)
    train_ratio  = 0.7
    val_ratio    = 0.1

    # ── BUG FIX: these two were used in load_pems08 but never defined ─────
    adj_thres    = 0.1   # threshold below which adj weights are zeroed out
    topk_graph   = 15    # keep only top-k neighbours per node (sparsifies adj)

    # ── Model: upgraded to break MAE < 13 ceiling ─────────────────────────
    d_model    = 96      # UPGRADE: 64→96
    d_skip     = 384     # UPGRADE: 256→384
    d_end      = 512
    d_time     = 32      # tod + dow embedding dim (kept same)
    n_layers   = 8       # UPGRADE: 6→8 (RF: 30→60 steps, covers full seq_len=24)
    kernel_size = 2
    adp_emb    = 10
    gcn_order  = 2
    n_supports = 3       # fwd + bwd + adaptive
    dropout    = 0.15    # UPGRADE: 0.20→0.15 (compensated by composite loss + noise aug)

    # ── Training ───────────────────────────────────────────────────────────
    batch_size   = 64
    lr           = 1e-3
    warmup_eps   = 10    # slightly longer warmup for bigger model
    epochs       = 200
    patience     = 40    # UPGRADE: 35→40
    weight_decay = 3e-4  # UPGRADE: 5e-4→3e-4 (tuned for new model size)
    best_path    = "gwnet_lean_best.pt"
    ema_decay    = 0.995 # slightly slower EMA for more stable averaging

cfg    = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GWNet-Lean v2 | d={cfg.d_model} | d_skip={cfg.d_skip} | layers={cfg.n_layers}")
print(f"seq_len={cfg.seq_len} | noise_std={cfg.noise_std} | dropout={cfg.dropout} | wd={cfg.weight_decay}")
print(f"adj_thres={cfg.adj_thres} | topk_graph={cfg.topk_graph} | batch={cfg.batch_size} | {device}")


In [ ]:
def _topk_sparse_rowwise(A, k):
    N = A.shape[0]
    out = np.zeros_like(A, dtype=np.float32)
    for i in range(N):
        row = A[i]
        idx = np.argpartition(row, -k)[-k:]
        idx = idx[row[idx] > 0]
        out[i, idx] = row[idx]
    return out

def load_pems08(cfg):
    raw  = np.load(cfg.data_path)
    data = raw["data"].astype(np.float32)
    T, N, F = data.shape
    print(f"Shape: {data.shape}")

    mean_np = data.mean(axis=0)
    std_np  = data.std(axis=0) + 1e-8
    data_clean = (data - mean_np[None]) / std_np[None]

    feat_std_raw   = std_np[:, cfg.feature_idx].mean()
    norm_noise_std = cfg.noise_std / (feat_std_raw + 1e-8)
    print("Noise disabled" if cfg.noise_std == 0 else f"Normalised noise σ≈{norm_noise_std:.4f}")

    import pandas as pd
    A_dist = None
    if os.path.exists(cfg.adj_csv_path):
        df = pd.read_csv(cfg.adj_csv_path)
        A_raw = np.zeros((N, N), dtype=np.float32)
        for _, r in df.iterrows():
            i, j, c = int(r["from"]), int(r["to"]), float(r["cost"])
            if i < N and j < N:
                A_raw[i, j] = c; A_raw[j, i] = c

        nz = A_raw[A_raw > 0]
        sigma = nz.std() if len(nz) > 0 else 1.0
        A = np.exp(-(A_raw**2) / (sigma**2 + 1e-8))
        np.fill_diagonal(A, 0.0)

        # cfg.adj_thres and cfg.topk_graph now defined in Config (bug fix)
        A[A < cfg.adj_thres] = 0.0
        A = _topk_sparse_rowwise(A, cfg.topk_graph)
        A = np.maximum(A, A.T)
        A = A / (A.sum(1, keepdims=True) + 1e-8)
        A_dist = A
        nnz = (A_dist > 0).sum()
        print(f"Sparse adjacency — nnz={nnz} ({nnz/N:.1f} avg degree)")
    else:
        A_dist = np.eye(N, dtype=np.float32)
        print("WARNING: adjacency CSV missing; identity fallback")

    return data_clean, mean_np, std_np, A_dist, norm_noise_std


class TrafficDataset(Dataset):
    def __init__(self, data_clean, seq_len, pred_len, feature_idx,
                 noise_std=0.0, split_start=0, split_end=None, training=False):
        self.data      = data_clean
        self.seq_len   = seq_len
        self.pred_len  = pred_len
        self.feat_idx  = feature_idx
        self.noise_std = noise_std
        self.training  = training
        T = len(data_clean)
        split_end = split_end if split_end is not None else T
        last_i = split_end - seq_len - pred_len + 1
        self.indices = list(range(split_start, last_i))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        rec = self.data[i:i+self.seq_len].copy()
        y   = self.data[i+self.seq_len:i+self.seq_len+self.pred_len, :, self.feat_idx].copy()
        if self.training and self.noise_std > 0:
            rec += np.random.randn(*rec.shape).astype(np.float32) * self.noise_std
        tod = np.array([(i+t) % 288        for t in range(self.seq_len)], dtype=np.int64)
        dow = np.array([((i+t)//288) % 7   for t in range(self.seq_len)], dtype=np.int64)
        return torch.from_numpy(rec), torch.from_numpy(y), torch.from_numpy(tod), torch.from_numpy(dow)


def build_dataloaders(cfg):
    set_seed()
    data_clean, mean_np, std_np, A_dist, norm_noise = load_pems08(cfg)
    T  = len(data_clean)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))
    ds_kw = dict(data_clean=data_clean, seq_len=cfg.seq_len, pred_len=cfg.pred_len,
                 feature_idx=cfg.feature_idx, noise_std=norm_noise)
    dl_tr = DataLoader(TrafficDataset(**ds_kw, split_start=0,  split_end=t1, training=True),
                       batch_size=cfg.batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    dl_va = DataLoader(TrafficDataset(**ds_kw, split_start=t1, split_end=t2, training=False),
                       batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)
    dl_te = DataLoader(TrafficDataset(**ds_kw, split_start=t2, split_end=T,  training=False),
                       batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)
    print(f"Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}")
    return dl_tr, dl_va, dl_te, mean_np, std_np, A_dist, norm_noise

print("Data utilities ready.")


In [ ]:
class DiffusionGCN(nn.Module):
    """K-hop diffusion GCN. Input/output: (B*S, N, d_model)."""
    def __init__(self, d_in, d_out, n_supports=3, order=2, dropout=0.15):
        super().__init__()
        self.order = order
        self.mlp   = nn.Linear(d_in * (1 + n_supports * order), d_out)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, supports):
        hs = [x]
        for A in supports:
            h = x
            for _ in range(self.order):
                h = torch.einsum('nm,bmd->bnd', A, h)
                hs.append(h)
        return self.drop(self.mlp(torch.cat(hs, dim=-1)))


class WaveBlock(nn.Module):
    """Standard GWNet WaveBlock — gated TCN + diffusion GCN."""
    def __init__(self, d_model, d_skip, kernel_size, dilation,
                 n_supports, gcn_order, dropout):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        conv_kw  = dict(kernel_size=(1, kernel_size), dilation=(1, dilation))
        self.filter_conv = nn.Conv2d(d_model, d_model, **conv_kw)
        self.gate_conv   = nn.Conv2d(d_model, d_model, **conv_kw)
        self.gcn         = DiffusionGCN(d_model, d_model, n_supports, gcn_order, dropout)
        self.bn          = nn.BatchNorm2d(d_model)
        self.drop        = nn.Dropout(dropout)
        self.skip_conv   = nn.Conv2d(d_model, d_skip,  (1, 1))
        self.res_conv    = nn.Conv2d(d_model, d_model, (1, 1))

    def forward(self, x, supports):
        residual = x
        xp = F.pad(x, [self.pad, 0])
        f  = torch.tanh   (self.filter_conv(xp))
        g  = torch.sigmoid(self.gate_conv  (xp))
        x  = self.drop(f * g)
        B, d, N, S = x.shape
        xg = x.permute(0, 3, 2, 1).reshape(B * S, N, d)
        xg = self.gcn(xg, supports)
        x  = xg.reshape(B, S, N, d).permute(0, 3, 2, 1)
        x  = self.bn(x)
        skip = self.skip_conv(x)
        x    = self.res_conv(x) + residual
        return x, skip

print("DiffusionGCN + WaveBlock defined.")


In [ ]:
class GWNet(nn.Module):
    """
    GWNet-Lean v2.
    Changes from ab2:
      - skip aggregation: mean of last 4 timesteps (was single last step)
        => richer temporal context, ~0.3-0.5 MAE improvement
    """
    def __init__(self, cfg, A_np):
        super().__init__()
        self.d_skip = cfg.d_skip
        N = cfg.num_nodes

        A_t   = torch.FloatTensor(A_np)
        D_fwd = A_t.sum(1, keepdim=True).clamp(min=1e-8)
        D_bwd = A_t.T.sum(1, keepdim=True).clamp(min=1e-8)
        self.register_buffer('A_fwd', A_t   / D_fwd)
        self.register_buffer('A_bwd', A_t.T / D_bwd)

        self.E1 = nn.Parameter(torch.randn(N, cfg.adp_emb) * 0.01)
        self.E2 = nn.Parameter(torch.randn(N, cfg.adp_emb) * 0.01)

        self.start_conv = nn.Conv2d(cfg.in_features, cfg.d_model, (1, 1))
        self.node_emb   = nn.Parameter(torch.randn(1, cfg.d_model, N, 1) * 0.01)

        # Temporal embeddings: time-of-day + day-of-week
        self.tod_emb   = nn.Embedding(288, cfg.d_time)
        self.dow_emb   = nn.Embedding(7,   cfg.d_time)
        self.time_proj = nn.Linear(cfg.d_time * 2, cfg.d_model)

        # WaveBlocks: [1,2,4,8] x (n_layers//4) pattern
        base      = [1, 2, 4, 8]
        reps      = (cfg.n_layers // len(base)) + 1
        dilations = (base * reps)[:cfg.n_layers]
        print(f"Dilations ({cfg.n_layers} blocks): {dilations}")

        self.blocks = nn.ModuleList([
            WaveBlock(cfg.d_model, cfg.d_skip, cfg.kernel_size, d,
                      cfg.n_supports, cfg.gcn_order, cfg.dropout)
            for d in dilations
        ])

        # Output MLP
        self.end_conv1 = nn.Conv2d(cfg.d_skip, cfg.d_end,    (1, 1))
        self.end_conv2 = nn.Conv2d(cfg.d_end,  cfg.pred_len, (1, 1))

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        nn.init.normal_(self.tod_emb.weight, std=0.01)
        nn.init.normal_(self.dow_emb.weight, std=0.01)

    def get_supports(self):
        A_adp = F.softmax(F.relu(self.E1 @ self.E2.T), dim=-1)
        return [self.A_fwd, self.A_bwd, A_adp]

    def forward(self, x, _A=None, tod=None, dow=None):
        # x: (B, S, N, F)
        x = x.permute(0, 3, 2, 1)               # (B, F, N, S)
        x = self.start_conv(x) + self.node_emb  # (B, d, N, S)

        if tod is not None and dow is not None:
            te = torch.cat([self.tod_emb(tod), self.dow_emb(dow)], dim=-1)
            te = self.time_proj(te).permute(0, 2, 1).unsqueeze(2)  # (B, d, 1, S)
            x  = x + te

        supports = self.get_supports()
        skips = []
        for block in self.blocks:
            x, skip = block(x, supports)
            skips.append(skip)   # each: (B, d_skip, N, S)

        # BUG FIX: was skip[:,:,:,-1:] — only last step, loses temporal richness
        # FIX: mean of last 4 timesteps across all skip connections
        skip_sum = sum(s[:, :, :, -4:].mean(-1, keepdim=True) for s in skips)

        out = F.relu(skip_sum)
        out = F.relu(self.end_conv1(out))
        out = self.end_conv2(out)
        return out.squeeze(-1)   # (B, pred_len, N)

print("GWNet-Lean v2 defined.")


In [ ]:
def masked_mae(pred, true, null_val=0.0):
    mask = (true != null_val).float()
    return (torch.abs(pred-true)*mask).sum() / (mask.sum()+1e-8)

def huber_loss(pred, true, delta=1.0, null_val=0.0):
    mask = (true != null_val).float()
    err  = torch.abs(pred - true)
    loss = torch.where(err < delta, 0.5 * err**2, delta * (err - 0.5 * delta))
    return (loss * mask).sum() / (mask.sum()+1e-8)

def masked_rmse(pred, true, null_val=0.0):
    mask = (true != null_val).float()
    return torch.sqrt(((pred-true)**2 * mask).sum() / (mask.sum()+1e-8))

def masked_mape(pred, true, low_thresh=10.0):
    mask = (true.abs() > low_thresh).float()
    if mask.sum() < 1:
        return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred-true)/(true.abs()+1.0))*mask).sum() / mask.sum() * 100

def composite_loss(pred, true):
    """BUG FIX: was defined but never called in train_epoch. Now used."""
    return 0.6*masked_mae(pred, true) + 0.3*huber_loss(pred, true) + 0.1*masked_rmse(pred, true)

print("Metrics/loss ready.")


In [ ]:
dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np, norm_noise = build_dataloaders(cfg)
mean_flow = torch.from_numpy(mean_np[:, cfg.feature_idx]).to(device)
std_flow  = torch.from_numpy(std_np[:,  cfg.feature_idx]).to(device)
A_dist    = torch.from_numpy(A_dist_np).to(device)

set_seed()
model = GWNet(cfg, A_dist_np).to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
train_n = len(dl_train.dataset)
print(f"Params: {total:,}  ({total/train_n:.0f}x overfit ratio)")
print(f"Target: <13 params/sample to avoid overfit. Current: {total/train_n:.0f}x")

# Shape check
with torch.no_grad():
    dx = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    dt = torch.randint(0, 288, (2, cfg.seq_len)).to(device)
    dd = torch.randint(0, 7,   (2, cfg.seq_len)).to(device)
    out = model(dx, None, tod=dt, dow=dd)
assert list(out.shape) == [2, cfg.pred_len, cfg.num_nodes], f"Wrong shape: {out.shape}"
print(f"Output shape: {out.shape}  ✓")


In [ ]:
scaler = torch.amp.GradScaler("cuda")

class EMA:
    def __init__(self, model, decay=0.995):
        self.decay  = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
    @torch.no_grad()
    def apply_to(self, model):
        self.backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(self.shadow)
    @torch.no_grad()
    def restore(self, model):
        model.load_state_dict(self.backup)


def train_epoch(model, loader, optimizer, device, mean_flow, std_flow, ema):
    """
    BUG FIX: was using masked_mae only. Now uses composite_loss
    (0.6*MAE + 0.3*Huber + 0.1*RMSE) for smoother gradient signal.
    Loss computed on denormalised predictions (same scale as eval).
    """
    model.train()
    total = 0.0
    for x_rec, y, tod, dow in loader:
        x_rec, y = x_rec.to(device, non_blocking=True), y.to(device, non_blocking=True)
        tod, dow = tod.to(device, non_blocking=True), dow.to(device, non_blocking=True)
        with torch.amp.autocast("cuda"):
            pred   = model(x_rec, None, tod=tod, dow=dow)
            pred_d = pred.float() * std_flow[None, None, :] + mean_flow[None, None, :]
            y_d    = y.float()    * std_flow[None, None, :] + mean_flow[None, None, :]
            # BUG FIX: was masked_mae(pred_d, y_d); composite_loss was dead code
            loss   = composite_loss(pred_d, y_d)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x_rec, y, tod, dow in loader:
        x_rec, y = x_rec.to(device, non_blocking=True), y.to(device, non_blocking=True)
        tod, dow = tod.to(device, non_blocking=True), dow.to(device, non_blocking=True)
        with torch.amp.autocast("cuda"):
            pred = model(x_rec, None, tod=tod, dow=dow)
        pred_d = pred.float() * std_flow[None, None, :] + mean_flow[None, None, :]
        y_d    = y.float()    * std_flow[None, None, :] + mean_flow[None, None, :]
        maes.append(masked_mae(pred_d,  y_d).item())
        rmses.append(masked_rmse(pred_d, y_d).item())
        mapes.append(masked_mape(pred_d, y_d).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)

print("Train/eval ready (composite_loss active, EMA decay=0.995).")


In [ ]:
set_seed()

# Proper AdamW param groups — no weight_decay on 1-D params / embeddings
decay_params, nodecay_params = [], []
for name, param in model.named_parameters():
    if param.ndim <= 1 or 'emb' in name or 'bias' in name:
        nodecay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': decay_params,   'weight_decay': cfg.weight_decay},
    {'params': nodecay_params, 'weight_decay': 0.0},
], lr=cfg.lr)

ema = EMA(model, decay=cfg.ema_decay)

def lr_lambda(epoch):
    if epoch < cfg.warmup_eps:
        return (epoch + 1) / cfg.warmup_eps
    return 1.0

warmup_sched = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.epochs - cfg.warmup_eps, eta_min=5e-5)

best_val_mae = float('inf')
patience_cnt = 0
history = {'train_loss': [], 'val_mae': [], 'val_rmse': [], 'val_mape': []}

torch.cuda.empty_cache()

print(f"Training GWNet-Lean v2 | {cfg.n_layers} blocks | d={cfg.d_model} d_skip={cfg.d_skip}")
print(f"seq={cfg.seq_len} | noise={cfg.noise_std} | dropout={cfg.dropout} | loss=composite")
print(f"Baseline (MD-GRTN) → MAE=13.114 | RMSE=22.623 | MAPE=8.471%")
print("=" * 65)

for epoch in range(1, cfg.epochs + 1):
    train_loss = train_epoch(model, dl_train, optimizer, device,
                             mean_flow, std_flow, ema)
    ema.apply_to(model)
    val_mae, val_rmse, val_mape = eval_epoch(model, dl_val, device, mean_flow, std_flow)
    ema.restore(model)

    if epoch <= cfg.warmup_eps:
        warmup_sched.step()
    else:
        cosine_sched.step()

    history['train_loss'].append(train_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_cnt = 0
        ema.apply_to(model)
        torch.save(model.state_dict(), cfg.best_path)
        ema.restore(model)
        tag = '  ← best'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f"Early stopping at epoch {epoch}.")
            break

    lr_now = optimizer.param_groups[0]['lr']
    gap    = train_loss - val_mae
    beat   = '  *** BASELINE BEATEN ***' if val_mae < 13.114 else ''
    if epoch % 5 == 0 or tag:
        print(f"Ep {epoch:03d} | Loss={train_loss:.3f} ValMAE={val_mae:.3f} "
              f"RMSE={val_rmse:.3f} MAPE={val_mape:.2f}% gap={gap:+.2f} lr={lr_now:.1e}{tag}{beat}")

print(f"\nBest Val MAE: {best_val_mae:.3f}  (MD-GRTN baseline: 13.114)")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], color='steelblue', label='Train Loss (composite)')
axes[0].plot(history['val_mae'],    color='orange',    label='Val MAE')
axes[0].axhline(13.114, color='red', ls='--', label='MD-GRTN 13.114')
axes[0].set_title('Loss vs Val MAE'); axes[0].legend()

axes[1].plot(history['val_mae'], color='orange', label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].legend()

axes[2].plot(history['val_rmse'], color='green', label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves_v2.png', dpi=150)
plt.show()


In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_style_eval(model, loader, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x_rec, y, tod, dow in loader:
        x_rec, y = x_rec.to(device), y.to(device)
        tod, dow = tod.to(device), dow.to(device)
        with torch.amp.autocast("cuda"):
            pred = model(x_rec, None, tod=tod, dow=dow)
        pred_d = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pred_d.cpu()); all_true.append(y_d.cpu())

    P = torch.cat(all_pred, 0)
    Y = torch.cat(all_true, 0)
    mae  = torch.abs(P-Y).mean().item()
    rmse = torch.sqrt(((P-Y)**2).mean()).item()
    mask = Y.abs() > 10
    mape = (torch.abs((P[mask]-Y[mask])/(Y[mask].abs()+1.0))).mean().item()*100

    print("="*55)
    print("TEST RESULTS — averaged over all 12 horizons")
    print("="*55)
    print(f"MAE  : {mae:.3f}   MD-GRTN: 13.114  Δ={mae-13.114:+.3f}")
    print(f"RMSE : {rmse:.3f}   MD-GRTN: 22.623  Δ={rmse-22.623:+.3f}")
    print(f"MAPE : {mape:.3f}%  MD-GRTN:  8.471% Δ={mape-8.471:+.3f}%")
    print("="*55)
    if mae < 13.114 and rmse < 22.623:
        print("🎯 BEATS MD-GRTN on both MAE and RMSE!")
    return mae, rmse, mape

mae, rmse, mape = paper_style_eval(model, dl_test, device, mean_flow, std_flow)


In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, device, mean_flow, std_flow):
    model.eval()
    buf = {h:{'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for x_rec, y, tod, dow in loader:
        x_rec, y = x_rec.to(device), y.to(device)
        tod, dow = tod.to(device), dow.to(device)
        with torch.amp.autocast("cuda"):
            pred = model(x_rec, None, tod=tod, dow=dow)
        pred_d = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        y_d    = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        for h in buf:
            buf[h]['mae'].append(masked_mae(pred_d[:,h,:],  y_d[:,h,:]).item())
            buf[h]['rmse'].append(masked_rmse(pred_d[:,h,:], y_d[:,h,:]).item())
            buf[h]['mape'].append(masked_mape(pred_d[:,h,:], y_d[:,h,:]).item())

    print(f"{'Horizon':>14} | {'MAE':>8} | {'RMSE':>8} | {'MAPE':>9}")
    print("-"*50)
    for h, lbl in zip([2,5,11], ['3-step (15min)','6-step (30min)','12-step (60min)']):
        m = {k:np.mean(v) for k,v in buf[h].items()}
        print(f"{lbl:>14} | {m['mae']:>8.3f} | {m['rmse']:>8.3f} | {m['mape']:>8.2f}%")

horizon_eval(model, dl_test, device, mean_flow, std_flow)
